In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import tensorflow as tf
from keras import layers, models, losses, regularizers
from keras.models import load_model
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print("GPU Trovate:", len(tf.config.list_physical_devices('GPU')))
for gpu in tf.config.list_physical_devices('GPU'):
    print("Nome:", gpu.name)

# 1. SETUP MEMORIA: DEVE ESSERE LA PRIMA COSA IN ASSOLUTO
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memoria GPU configurata in modalità dinamica.")
    except RuntimeError as e:
        print("Errore GPU:", e)

# 2. ESORCISMO DELLA RAM (Uccide i vecchi modelli in memoria)
tf.keras.backend.clear_session()

# ABILITA IL GPU MEMORY GROWTH (Evita il crash VRAM)
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory Growth abilitato con successo.")
    except RuntimeError as e:
        print(f"Errore configurazione GPU: {e}")

I0000 00:00:1784039751.500186    7601 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPU Trovate: 1
Nome: /physical_device:GPU:0
Memoria GPU configurata in modalità dinamica.
✅ GPU Memory Growth abilitato con successo.


In [2]:
def embedded_summary(model, input_shape=(1, 120, 18), is_int8=False):
    total_params = model.count_params()
    
    # 1. Calcolo FLASH (4 byte per Float32, 1 byte per INT8)
    bytes_per_param = 1 if is_int8 else 4
    estimated_flash_kb = (total_params * bytes_per_param) / 1024
    
    # 2. Calcolo SRAM (Tensor Arena) con logica Adiacente (Buffer Reuse)
    bytes_per_activation = 1 if is_int8 else 4
    max_adjacent_ram_kb = 0
    
    # Memoria occupata dal layer precedente (inizializzata con la dimensione dell'input)
    previous_layer_size = (np.prod(input_shape) * bytes_per_activation) / 1024
    
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            # Per i layer come "Concatenate" che potrebbero avere output multipli/strani
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        current_layer_size = (num_elements * bytes_per_activation) / 1024
        
        # IL FIX È QUI: Sommiamo il layer precedente e il layer corrente!
        # È il momento esatto in cui TFLM consuma più RAM durante l'esecuzione di questo layer.
        current_peak = previous_layer_size + current_layer_size
        
        if current_peak > max_adjacent_ram_kb:
            max_adjacent_ram_kb = current_peak
            
        previous_layer_size = current_layer_size

    print("============================================")
    mode_str = "INT8 (Quantizzato)" if is_int8 else "FLOAT32 (Training)"
    print(f"   REPORT REQUISITI ESP32-S3 [{mode_str}]   ")
    print("============================================")
    print(f" Memoria FLASH stimata : ~{estimated_flash_kb:.2f} KB  (Limite: 800 KB)")
    print(f" Memoria SRAM stimata  : ~{max_adjacent_ram_kb:.2f} KB (Limite: 300 KB)")
    print("============================================\n")

In [3]:
# =====================================================================
# BULGARIAN SQUAT PER MAC BEST
# =====================================================================
import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)
ROOM_DIMS = tf.constant([4.8, 7.2], dtype=tf.float32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (1.5 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1)) 
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1)) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

I0000 00:00:1784039758.297194    7601 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2547 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


In [4]:
# ==============================================================================
# DATA ENGINE (Caricamento Globale in RAM + normalizzazione [0, 1]) best
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.02):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy'].astype(np.float32) # Assicuriamoci sia float
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato.")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y


# ==============================================================================
# 2. SPLIT DIVERSI SEGUENDO DIVERSI CRITERI
# ==============================================================================
# Split1 (78% train e 22% val), > windows con 3/4 persone in train
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]
# Split2 (80% train e 22% val), stesso cocetto di split 1 "STRESS TEST sul MULTIPATH"
#val_indices = [23, 20, 0, 13, 9] 
#train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]
# Split3 (78% train e 22% val) > equilibrato tra train e val 
#val_indices = [22, 20, 2, 12, 14, 6] 
#train_indices = [23, 16, 18, 19, 21, 0, 1, 3, 4, 13, 15, 5, 7, 8, 9, 10, 11, 17]

tutti_i_file = glob.glob("dataset/data/*.npz")
#tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 2. CARICAMENTO DATI GREZZI
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train_raw, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val_raw, Y_val = load_and_process_all_files(val_files)

# ==============================================================================
# 3. NORMALIZZAZIONE X (INPUT) CON GLOBAL MAX
# ==============================================================================
GLOBAL_MAX = np.percentile(X_train_raw, 99.5)
print(f"\n---> GLOBAL_MAX CALCOLATO: {GLOBAL_MAX:.2f} <---")
print("INSERISCI QUESTO VALORE NEL TUO CODE.PY PER L'INFERENZA!")

X_train = np.clip(X_train_raw, 0, GLOBAL_MAX) / GLOBAL_MAX
X_val = np.clip(X_val_raw, 0, GLOBAL_MAX) / GLOBAL_MAX

print("\n==================================================")
print(f"DATI TOTALI PRONTI E NORMALIZZATI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 18 file...
File 1/18 processato.
File 2/18 processato.
File 3/18 processato.
File 4/18 processato.
File 5/18 processato.
File 6/18 processato.
File 7/18 processato.
File 8/18 processato.
File 9/18 processato.
File 10/18 processato.
File 11/18 processato.
File 12/18 processato.
File 13/18 processato.
File 14/18 processato.
File 15/18 processato.
File 16/18 processato.
File 17/18 processato.
File 18/18 processato.

--- PREPARAZIONE VALIDATION SET ---
Inizio caricamento ed EMA Decluttering di 6 file...
File 1/6 processato.
File 2/6 processato.
File 3/6 processato.
File 4/6 processato.
File 5/6 processato.
File 6/6 processato.

---> GLOBAL_MAX CALCOLATO: 203.93 <---
INSERISCI QUESTO VALORE NEL TUO CODE.PY PER L'INFERENZA!

DATI TOTALI PRONTI E NORMALIZZATI IN RAM!
Totale FRAME individuali di Train:      135000
Totale FRAME individuali di Validation: 45000


In [ ]:
# ==============================================================================
# DATA ENGINE (Caricamento Globale in RAM + normalizzazione [0, 1] + Data Augmentation)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.02):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy'].astype(np.float32) # Assicuriamoci sia float
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato.")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y


# ==============================================================================
# 2. SPLIT DIVERSI SEGUENDO DIVERSI CRITERI
# ==============================================================================
# Split1 (78% train e 22% val), > windows con 3/4 persone in train
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]
# Split2 (80% train e 22% val), stesso cocetto di split 1 "STRESS TEST sul MULTIPATH"
#val_indices = [23, 20, 0, 13, 9] 
#train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]
# Split3 (78% train e 22% val) > equilibrato tra train e val 
#val_indices = [22, 20, 2, 12, 14, 6] 
#train_indices = [23, 16, 18, 19, 21, 0, 1, 3, 4, 13, 15, 5, 7, 8, 9, 10, 11, 17]

tutti_i_file = glob.glob("dataset/data/*.npz")
#tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 2. CARICAMENTO DATI GREZZI
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train_raw, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val_raw, Y_val = load_and_process_all_files(val_files)

# ==============================================================================
# 3. NORMALIZZAZIONE X (INPUT) CON GLOBAL MAX
# ==============================================================================
GLOBAL_MAX = np.percentile(X_train_raw, 99.5)
print(f"\n---> GLOBAL_MAX CALCOLATO: {GLOBAL_MAX:.2f} <---")
print("INSERISCI QUESTO VALORE NEL TUO CODE.PY PER L'INFERENZA!")

X_train = np.clip(X_train_raw, 0, GLOBAL_MAX) / GLOBAL_MAX
X_val = np.clip(X_val_raw, 0, GLOBAL_MAX) / GLOBAL_MAX

print("\n==================================================")
print(f"DATI TOTALI PRONTI E NORMALIZZATI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")

# ==============================================================================
# 3. FUNZIONE DI DATA AUGMENTATION (RADAR DROPOUT)
# ==============================================================================
def radar_dropout_augmentation(x, y):
    drop_rate = 0.20 # 20% di probabilità di spegnere un intero radar
    x_reshaped = tf.reshape(x, (1, 120, 6, 3))
    random_tensor = tf.random.uniform(shape=(1, 1, 6, 1))
    keep_mask = tf.cast(random_tensor >= drop_rate, tf.float32)
    x_augmented = (x_reshaped * keep_mask) / (1.0 - drop_rate)
    x_final = tf.reshape(x_augmented, (1, 120, 18))
    return x_final, y

# ==============================================================================
# 4. CREAZIONE PIPELINE TF.DATA (SOSTITUISCE LE MATRICI NUMPY NEL TRAINING)
# ==============================================================================
print("\n--- PREPARAZIONE TF.DATA PIPELINE CON RADAR DROPOUT ---")

# FIX MEMORIA GPU: Forziamo la permanenza del dataset gigante in RAM normale (CPU)
with tf.device('/CPU:0'):
    # Il Train Dataset riceve Shuffle, Radar Dropout (tramite .map) e Batching
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
    train_dataset = train_dataset.shuffle(buffer_size=5000)
    train_dataset = train_dataset.map(radar_dropout_augmentation, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
    
    # Il Validation Dataset NON riceve il dropout
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
    val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 18 file...
File 1/18 processato.
File 2/18 processato.
File 3/18 processato.
File 4/18 processato.
File 5/18 processato.
File 6/18 processato.
File 7/18 processato.
File 8/18 processato.
File 9/18 processato.
File 10/18 processato.
File 11/18 processato.
File 12/18 processato.
File 13/18 processato.
File 14/18 processato.
File 15/18 processato.
File 16/18 processato.
File 17/18 processato.
File 18/18 processato.

--- PREPARAZIONE VALIDATION SET ---
Inizio caricamento ed EMA Decluttering di 6 file...
File 1/6 processato.
File 2/6 processato.
File 3/6 processato.
File 4/6 processato.
File 5/6 processato.
File 6/6 processato.

---> GLOBAL_MAX CALCOLATO: 203.93 <---
INSERISCI QUESTO VALORE NEL TUO CODE.PY PER L'INFERENZA!

DATI TOTALI PRONTI E NORMALIZZATI IN RAM!
Totale FRAME individuali di Train:      135000
Totale FRAME individuali di Validation: 45000

--- PREPARAZIONE TF.DATA PIPELINE CON RADAR DROPOUT ---


W0000 00:00:1783526413.310899   46098 cpu_allocator_impl.cc:82] Allocation of 1166400000 exceeds 10% of free system memory.


In [5]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2 - RESIDUAL REDUCTION MODULES (RRM)
# ==============================================================================

def squeeze_excite_block_2d(x, filters, r=8):
    """Meccanismo di Attenzione spaziale basato su Squeeze-and-Excitation"""
    # Squeeze: estrae le statistiche globali per ogni canale
    se = layers.GlobalAveragePooling2D()(x)
    # Excitation: riduce e poi ri-espande per imparare i pesi ottimali
    se = layers.Dense(max(1, filters // r), activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    # Reshape per applicare il broadcasting moltiplicativo
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def residual_reduction_module_2d(x, filters, r=8, name_prefix=""):
    """
    RRM: Red(Res(x)) 
    Unisce una skip connection pesata dal SE block e un dimezzamento 
    parallelo della dimensione temporale (range bins).
    """
    # --- 1. Residual Branch (Res) ---
    res = layers.Conv2D(filters, kernel_size=(1, 3), padding='same', activation='relu', 
                        name=f"{name_prefix}_res_conv")(x)
    res = squeeze_excite_block_2d(res, filters, r=r)
    res = layers.Add(name=f"{name_prefix}_res_add")([res, x]) # Skip connection

    # --- 2. Reduction Branch (Red) ---
    # Due convoluzioni parallele con stride=(1,2) per dimezzare i range bins (da 120->60->30...)
    red1 = layers.Conv2D(filters, kernel_size=(1, 3), strides=(1, 2), padding='same', 
                         activation='relu', name=f"{name_prefix}_red_conv1")(res)
    # Kernel 1x1 funge da projection mapping tipico delle ResNet
    red2 = layers.Conv2D(filters, kernel_size=(1, 1), strides=(1, 2), padding='same', 
                         activation='relu', name=f"{name_prefix}_red_conv2")(res)
    
    out = layers.Add(name=f"{name_prefix}_red_add")([red1, red2])
    return out

def build_eeai_model_v2_rrm(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")
    
    F = 64 # Numero di filtri base: mantiene il modello piccolo e potente
    r = 8  # Reduction ratio per il blocco SE (come da paper)
    
    # 1. Feature Extraction Iniziale (allinea il numero di canali a F per far funzionare le Add)
    x = layers.Conv2D(F, kernel_size=(1, 5), padding='same', activation='relu', name="init_conv")(inputs)
    
    # 2. Cascata di Residual Reduction Modules
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm1") # Bins: 120 -> 60
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm2") # Bins: 60 -> 30
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm3") # Bins: 30 -> 15
    # Excitation: riduce e poi ri-espande per imparare
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm4") # Bins: 15 -> 8
    
    # 3. Testa della rete (Flatten + Dense)
    x = layers.Flatten(name="flatten_features")(x)
    x = layers.Dropout(0.25, name="dropout_features")(x)
    
    common_feat = layers.Dense(128, activation='relu', name="dense_shared")(x)
    
    # 4. Multi-Head Output (Coordinate + Maschera presenze)
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])
    
    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_RRM")

In [5]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2 - RESIDUAL REDUCTION MODULES (RRM) CON DEPTHWISE SEPARABLE CONV BEST
# ==============================================================================

def squeeze_excite_block_2d(x, filters, r=8):
    """Meccanismo di Attenzione spaziale basato su Squeeze-and-Excitation"""
    # Squeeze: estrae le statistiche globali per ogni canale
    se = layers.GlobalAveragePooling2D()(x)
    # Excitation: riduce e poi ri-espande per imparare i pesi ottimali
    se = layers.Dense(max(1, filters // r), activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    # Reshape per applicare il broadcasting moltiplicativo
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def residual_reduction_module_2d_mobile(x, filters, r=8, name_prefix=""):
    # --- 1. Residual Branch (Res) ---
    # Depthwise Separable invece di Conv2D standard
    res = layers.DepthwiseConv2D(kernel_size=(1, 3), padding='same', use_bias=False, name=f"{name_prefix}_res_dw")(x)
    res = layers.BatchNormalization(name=f"{name_prefix}_res_bn1")(res)
    res = layers.ReLU(name=f"{name_prefix}_res_relu1")(res)
    res = layers.Conv2D(filters, kernel_size=(1, 1), padding='same', activation='relu', name=f"{name_prefix}_res_pw")(res)
    
    res = squeeze_excite_block_2d(res, filters, r=r)
    res = layers.Add(name=f"{name_prefix}_res_add")([res, x])

    # --- 2. Reduction Branch (Red) ---
    red1 = layers.DepthwiseConv2D(kernel_size=(1, 3), strides=(1, 2), padding='same', use_bias=False, name=f"{name_prefix}_red_dw")(res)
    red1 = layers.BatchNormalization(name=f"{name_prefix}_red_bn2")(red1)
    red1 = layers.ReLU(name=f"{name_prefix}_red_relu2")(red1)
    red1 = layers.Conv2D(filters, kernel_size=(1, 1), padding='same', activation='relu', name=f"{name_prefix}_red_pw")(red1)
    
    # red2 resta una Conv2D standard 1x1 (è già il metodo più economico)
    red2 = layers.Conv2D(filters, kernel_size=(1, 1), strides=(1, 2), padding='same', activation='relu', name=f"{name_prefix}_red_conv2")(res)
    
    out = layers.Add(name=f"{name_prefix}_red_add")([red1, red2])
    return out

def build_eeai_model_v2_rrm(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")
    
    F = 64 # Numero di filtri base: mantiene il modello piccolo e potente
    r = 8  # Reduction ratio per il blocco SE (come da paper)
    
    # Aggiunta di rumore gaussiano per regolarizzazione
    x = layers.GaussianNoise(0.01, name="input_noise")(inputs)

    # 1. Feature Extraction Iniziale (allinea il numero di canali a F per far funzionare le Add)
    x = layers.Conv2D(F, kernel_size=(1, 5), padding='same', activation='relu', name="init_conv")(x)
    
    # 2. Cascata di Residual Reduction Modules
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm1") # Bins: 120 -> 60
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm2") # Bins: 60 -> 30
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm3") # Bins: 30 -> 15
    # Excitation: riduce e poi ri-espande per imparare
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm4") # Bins: 15 -> 8
    
    # 3. Testa della rete (Flatten + Dense)
    x = layers.Flatten(name="flatten_features")(x)
    x = layers.Dropout(0.35, name="dropout_features")(x)
    
    common_feat = layers.Dense(128, activation='relu', name="dense_shared")(x)
    
    # 4. Multi-Head Output (Coordinate + Maschera presenze)
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])
    
    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_RRM")

In [7]:
print(np.max(X_train))
print(np.mean(X_train))

1.0
0.060043737


In [7]:
# ==============================================================================
# ADDESTRAMENTO MODELLO V2 (RRM) BEST
# ==============================================================================

print("\n--- PREPARAZIONE TF.DATA PIPELINE ---")
# CREIAMO I DATASET (Previene la creazione di un singolo tensore gigante)
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
# Shuffle, batch e prefetch massimizzano l'uso della GPU senza saturarla
train_dataset = train_dataset.shuffle(buffer_size=5000).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

# Inizializzazione del nuovo modello
model_rrm = build_eeai_model_v2_rrm()

# Compilazione 
model_rrm.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_rrm = ModelCheckpoint("split1.keras", monitor="val_loss", save_best_only=True, verbose=1)
#reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# FUOCO ALLE POLVERI
EPOCHS = 150
embedded_summary(model_rrm, input_shape=(1, 120, 18), is_int8=False)

print("\n--- INIZIO ADDESTRAMENTO MODELLO RRM ---")
history_rrm = model_rrm.fit(
    train_dataset,
    validation_data= val_dataset,
    # batch_size=32,
    # shuffle=True,
    epochs=EPOCHS,
    callbacks=[checkpoint_rrm, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")


--- PREPARAZIONE TF.DATA PIPELINE ---


W0000 00:00:1784041961.768760    7601 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.09GiB (rounded to 1166400000)requested by op _EagerConst
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
I0000 00:00:1784041961.773255    7601 bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
I0000 00:00:1784041961.773262    7601 bfc_allocator.cc:1056] Bin (256): 	Total Chunks: 268, Chunks in use: 267. 67.0KiB allocated for chunks. 66.8KiB in use in bin. 26.6KiB client-requested in use in bin.
I0000 00:00:1784041961.773270    7601 bfc_allocator.cc:1056] Bin (512): 	Total Chunks: 26, Chunks in use: 24. 18.5KiB allocated for chunks. 17.5KiB in use in bin. 17.2KiB client-requested in use in bin.
I0000 00:00:1784041961.773273    7601 bfc_allocator.cc:1056] Bin (1024): 	Total Chunks: 5, Chunks in use: 4. 5.8KiB

KeyboardInterrupt: 

In [13]:
# ==============================================================================
# ADDESTRAMENTO MODELLO V2 (RRM) 
# ==============================================================================

print("\n--- PREPARAZIONE TF.DATA PIPELINE ---")
# CREIAMO I DATASET (Previene la creazione di un singolo tensore gigante)
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
# Shuffle, batch e prefetch massimizzano l'uso della GPU senza saturarla
train_dataset = train_dataset.shuffle(buffer_size=5000).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

# Inizializzazione del nuovo modello
model_rrm = build_eeai_model_v2_rrm()

# Compilazione 
model_rrm.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_rrm = ModelCheckpoint("norm+depth+noise_split2_model_toscano.keras", monitor="val_loss", save_best_only=True, verbose=1)
#reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# FUOCO ALLE POLVERI
EPOCHS = 100
embedded_summary(model_rrm, input_shape=(1, 120, 18), is_int8=False)

print("\n--- INIZIO ADDESTRAMENTO MODELLO RRM SPLIT DIVERSO---")
history_rrm = model_rrm.fit(
    train_dataset,
    validation_data= val_dataset,
    # batch_size=32,
    # shuffle=True,
    epochs=EPOCHS,
    callbacks=[checkpoint_rrm, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")


--- PREPARAZIONE TF.DATA PIPELINE ---


W0000 00:00:1783602313.029353   30216 cpu_allocator_impl.cc:82] Allocation of 1231200000 exceeds 10% of free system memory.
W0000 00:00:1783602314.317773   30216 cpu_allocator_impl.cc:82] Allocation of 1231200000 exceeds 10% of free system memory.


   REPORT REQUISITI ESP32-S3 [FLOAT32 (Training)]   
 Memoria FLASH stimata : ~510.30 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~0.00 KB (Limite: 300 KB)


--- INIZIO ADDESTRAMENTO MODELLO RRM SPLIT DIVERSO---
Epoch 1/100


W0000 00:00:1783602316.392007   30216 cpu_allocator_impl.cc:82] Allocation of 1231200000 exceeds 10% of free system memory.
I0000 00:00:1783602321.693349   30286 service.cc:153] XLA service 0x7a6b30053510 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1783602321.693377   30286 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce GTX 1650, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.10.1)
I0000 00:00:1783602321.860802   30286 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1783602322.935269   30286 cuda_dnn.cc:461] Loaded cuDNN version 91001
I0000 00:00:1783602323.095382   30286 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_13062__.103
I0000 00:00:1783602337.424091   30286 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


4452/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9599 - hungarian_rmse_metres: 0.6554 - loss: 0.7687

I0000 00:00:1783602366.446673   30284 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_13062__.103


4454/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - hungarian_mask_acc: 0.9599 - hungarian_rmse_metres: 0.6554 - loss: 0.7685

I0000 00:00:1783602381.481446   30284 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_36524__.20
I0000 00:00:1783602385.991115   30284 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_36524__.20



Epoch 1: val_loss improved from None to 0.83477, saving model to norm+depth+noise_split2_model_toscano.keras
4454/4454 ━━━━━━━━━━━━━━━━━━━━ 75s 12ms/step - hungarian_mask_acc: 0.9765 - hungarian_rmse_metres: 0.5443 - loss: 0.5409 - val_hungarian_mask_acc: 0.8806 - val_hungarian_rmse_metres: 0.5025 - val_loss: 0.8348 - learning_rate: 0.0010
Epoch 2/100
4449/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9887 - hungarian_rmse_metres: 0.4629 - loss: 0.3501
Epoch 2: val_loss improved from 0.83477 to 0.65670, saving model to norm+depth+noise_split2_model_toscano.keras
4454/4454 ━━━━━━━━━━━━━━━━━━━━ 29s 6ms/step - hungarian_mask_acc: 0.9873 - hungarian_rmse_metres: 0.4412 - loss: 0.3479 - val_hungarian_mask_acc: 0.9086 - val_hungarian_rmse_metres: 0.4276 - val_loss: 0.6567 - learning_rate: 0.0010
Epoch 3/100
4449/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9916 - hungarian_rmse_metres: 0.4104 - loss: 0.2819
Epoch 3: val_loss improved from 0.65670 to 0.52128,

In [ ]:
# 1. CARICA IL TUO GOLDEN MODEL (Assicurati di passare le tue loss custom!)
model_rrm = tf.keras.models.load_model(
    "best_model_toscano.keras", 
    custom_objects={
        'hungarian_total_loss': hungarian_total_loss,
        'hungarian_rmse_metres': hungarian_rmse_metres,
        'hungarian_mask_acc': hungarian_mask_acc
    }
)

# 2. RICOMPILA CON UN LEARNING RATE MICROSCOPICO
# Usiamo 1e-5 o 5e-6 per non distruggere i pesi già perfetti!
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)
model_rrm.compile(
    optimizer=optimizer,
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

# 3. IMPOSTA CALLBACKS MENO AGGRESSIVE
checkpoint_ft = tf.keras.callbacks.ModelCheckpoint("finetuned_model.keras", monitor="val_loss", save_best_only=True, verbose=1)
early_stop_ft = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# 4. LANCIA IL FINE-TUNING SUL DATASET CON IL RADAR DROPOUT
print("\n--- INIZIO FINE-TUNING CON RADAR DROPOUT ---")
history_ft = model_rrm.fit(
    train_dataset,               # La pipeline tf.data CON radar dropout al 15%
    validation_data=val_dataset, # La pipeline pulita
    epochs=30,                   # Bastano poche epoche
    callbacks=[checkpoint_ft, early_stop_ft], 
    verbose=1
)


--- INIZIO FINE-TUNING CON RADAR DROPOUT ---
Epoch 1/30


I0000 00:00:1783530462.514977   46169 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2872704__.103


4218/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - hungarian_mask_acc: 0.9922 - hungarian_rmse_metres: 0.3465 - loss: 0.2068

I0000 00:00:1783530544.995288   46170 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2872704__.103


4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - hungarian_mask_acc: 0.9922 - hungarian_rmse_metres: 0.3465 - loss: 0.2068

I0000 00:00:1783530553.534096   46170 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2894991__.20
I0000 00:00:1783530562.650825   46172 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2894991__.20



Epoch 1: val_loss improved from None to 0.50257, saving model to finetuned_model.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 111s 23ms/step - hungarian_mask_acc: 0.9908 - hungarian_rmse_metres: 0.3113 - loss: 0.1925 - val_hungarian_mask_acc: 0.9553 - val_hungarian_rmse_metres: 0.4256 - val_loss: 0.5026
Epoch 2/30
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - hungarian_mask_acc: 0.9908 - hungarian_rmse_metres: 0.3492 - loss: 0.2132
Epoch 2: val_loss did not improve from 0.50257
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 82s 19ms/step - hungarian_mask_acc: 0.9900 - hungarian_rmse_metres: 0.3117 - loss: 0.1949 - val_hungarian_mask_acc: 0.9550 - val_hungarian_rmse_metres: 0.4277 - val_loss: 0.5083
Epoch 3/30
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - hungarian_mask_acc: 0.9907 - hungarian_rmse_metres: 0.3491 - loss: 0.2136
Epoch 3: val_loss did not improve from 0.50257
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 82s 19ms/step - hungarian_mask_acc: 0.9898 - hungarian_rmse_metres: 0.3122 - loss: 0.1957 - val_hungarian_

In [11]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==============================================================================
# CONFIGURAZIONE
# ==============================================================================
# Inserisci i percorsi corretti per i tuoi file
FILE_TARGET = "/home/marco/Desktop/test_project_edge_ai/other_files/dataset/data/window_000021.npz"
KERAS_MODEL_PATH = "/home/marco/Desktop/test_project_edge_ai/other_files/best_model_toscano.keras"
TFLITE_MODEL_PATH = "/home/marco/Desktop/test_project_edge_ai/submission/model.tflite"

ALPHA = 0.02
GLOBAL_MAX = 203.93

if not os.path.exists(FILE_TARGET):
    print(f"ERRORE: Non trovo il file {FILE_TARGET}")
else:
    print("1️⃣ Caricamento Dati e Pre-processing (EMA + Normalizzazione)...")
    data = np.load(FILE_TARGET)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    
    for t in range(T):
        bg = ALPHA * mag[t] + (1 - ALPHA) * bg
        decluttered[t] = np.abs(mag[t] - bg)
        
    # NORMALIZZAZIONE FONDAMENTALE
    normalized_data = np.clip(decluttered, 0, GLOBAL_MAX) / GLOBAL_MAX

    # ==============================================================================
    # INFERENZA KERAS (FLOAT 32)
    # ==============================================================================
    print("2️⃣ Inferenza Modello Keras (Float32)...")
    model_keras = tf.keras.models.load_model(KERAS_MODEL_PATH, compile=False)
    preds_keras = model_keras.predict(normalized_data, verbose=0)
    
    k_coords = preds_keras[:, :8].reshape(T, 4, 2)
    k_mask = preds_keras[:, 8:]

    # ==============================================================================
    # INFERENZA TFLITE (INT 8)
    # ==============================================================================
    print("3️⃣ Inferenza Modello TFLite (INT8)... (Attendi qualche secondo)")
    interpreter = tf.lite.Interpreter(
        model_path=TFLITE_MODEL_PATH,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    )
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    in_scale, in_zp = input_details['quantization']
    out_scale, out_zp = output_details['quantization']

    tfl_coords = np.zeros((T, 4, 2))
    tfl_mask = np.zeros((T, 4))

    for t in range(T):
        input_float = np.expand_dims(normalized_data[t].astype(np.float32), axis=0)
        
        # Quantizza
        input_quant = np.round(input_float / in_scale) + in_zp
        input_quant = np.clip(input_quant, -128, 127).astype(np.int8)
        
        interpreter.set_tensor(input_details['index'], input_quant)
        interpreter.invoke()
        
        # Dequantizza
        preds_quant = interpreter.get_tensor(output_details['index'])[0]
        preds_float = (preds_quant.astype(np.float32) - out_zp) * out_scale
        
        tfl_coords[t] = preds_float[:8].reshape(4, 2)
        tfl_mask[t] = preds_float[8:]

    print("✅ Dati pronti! Inizializzazione Interfaccia Grafica...")

    # ==============================================================================
    # VISUALIZZATORE (IPYWIDGETS + MATPLOTLIB)
    # ==============================================================================
    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(6, 8)) # Leggermente più grande per farci stare tutto
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            ax.set_title(f"Confronto Keras vs TFLite | Window: 21 | Frame: {frame_idx}/{T-1}", fontsize=12, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            # 1. Disegna Ground Truth (Verde)
            for i in range(4):
                if gt_mask[frame_idx, i] > 0.5:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', alpha=0.4, label='Ground Truth' if i==0 else "")
                    ax.text(rx, ry + 0.25, f"GT_{i+1}", color='darkgreen', fontweight='bold', ha='center')

            # 2. Disegna Keras (Blu)
            for i in range(4):
                conf_k = float(k_mask[frame_idx, i])
                if conf_k >= soglia:
                    px, py = k_coords[frame_idx, i]
                    ax.scatter(px, py, c='blue', s=120, marker='s', edgecolors='darkblue', alpha=0.8, label='Pred Keras (Float32)' if i==0 else "")
                    ax.text(px, py - 0.25, f"K:{conf_k*100:.0f}%", color='blue', fontsize=9, ha='center', fontweight='bold')

            # 3. Disegna TFLite (Rosso)
            for i in range(4):
                conf_tfl = float(tfl_mask[frame_idx, i])
                if conf_tfl >= soglia:
                    px, py = tfl_coords[frame_idx, i]
                    # Disegniamo una X rossa un po' più piccola così se è sovrapposta si vede il quadrato blu sotto
                    ax.scatter(px, py, c='red', s=100, marker='X', edgecolors='darkred', alpha=0.9, label='Pred TFLite (INT8)' if i==0 else "")
                    ax.text(px, py - 0.45, f"T:{conf_tfl*100:.0f}%", color='red', fontsize=9, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True, fontsize=10)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    # Controlli (Uso un solo slider per la soglia per comodità)
    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:', layout=widgets.Layout(width='400px'))
    slider_soglia = widgets.FloatSlider(value=0.85, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    controls = widgets.VBox([slider_frame, slider_soglia])
    controls.layout.margin = '20px 20px 20px 0px' 
    ui = widgets.HBox([controls, out])
    
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

1️⃣ Caricamento Dati e Pre-processing (EMA + Normalizzazione)...
2️⃣ Inferenza Modello Keras (Float32)...


I0000 00:00:1783594183.265779   18477 service.cc:153] XLA service 0x7d5db803a700 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1783594183.265796   18477 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce GTX 1650, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.10.1)
I0000 00:00:1783594183.284028   18477 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1783594183.463580   18477 cuda_dnn.cc:461] Loaded cuDNN version 91001
I0000 00:00:1783594183.471176   18477 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2135__.5
I0000 00:00:1783594188.165988   18477 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
I0000 00:00:1783594189.621055   18480 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3200__.5


3️⃣ Inferenza Modello TFLite (INT8)... (Attendi qualche secondo)


/home/marco/yes/envs/edge_ai_env/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


✅ Dati pronti! Inizializzazione Interfaccia Grafica...


NORM: val_hungarian_mask_acc: 0.9487 - val_hungarian_rmse_metres: 0.4439
NORM + DEPTH: val_hungarian_mask_acc: 0.9411 - val_hungarian_rmse_metres: 0.4487
NORM + DEPTH + 0.35 DROPOUT: val_hungarian_mask_acc: 0.9479 - val_hungarian_rmse_metres: 0.4680
NORM + DEPTH + f=128 :val_hungarian_mask_acc: 0.9361 - val_hungarian_rmse_metres: 0.4729 
# BEST MODEL
NORM + DEPTH + O.25 DROPOUT + NOISE  val_hungarian_mask_acc: 0.9504 - val_hungarian_rmse_metres: 0.4136 


--- PREPARAZIONE TF.DATA PIPELINE ---
W0000 00:00:1783506948.791969   24474 cpu_allocator_impl.cc:82] Allocation of 1166400000 exceeds 10% of free system memory.
W0000 00:00:1783506949.993401   24474 cpu_allocator_impl.cc:82] Allocation of 1166400000 exceeds 10% of free system memory.
============================================
   REPORT REQUISITI ESP32-S3 [FLOAT32 (Training)]   
============================================
 Memoria FLASH stimata : ~510.30 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~0.00 KB (Limite: 300 KB)
============================================


--- INIZIO ADDESTRAMENTO MODELLO RRM ---
Epoch 1/100
W0000 00:00:1783506952.071505   24474 cpu_allocator_impl.cc:82] Allocation of 1166400000 exceeds 10% of free system memory.
I0000 00:00:1783506957.096045   24541 service.cc:153] XLA service 0x715c440586b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1783506957.096069   24541 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce GTX 1650, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.10.1)
I0000 00:00:1783506957.263911   24541 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1783506958.316876   24541 cuda_dnn.cc:461] Loaded cuDNN version 91001
I0000 00:00:1783506958.477087   24541 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_13060__.103
I0000 00:00:1783506972.559393   24541 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9627 - hungarian_rmse_metres: 0.7134 - loss: 0.8979
I0000 00:00:1783507000.223008   24541 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_13060__.103
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - hungarian_mask_acc: 0.9628 - hungarian_rmse_metres: 0.7133 - loss: 0.8976
I0000 00:00:1783507016.627744   24539 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_35347__.20
I0000 00:00:1783507021.374834   24537 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_35347__.20

Epoch 1: val_loss improved from None to 1.49275, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 75s 12ms/step - hungarian_mask_acc: 0.9756 - hungarian_rmse_metres: 0.5521 - loss: 0.5746 - val_hungarian_mask_acc: 0.8147 - val_hungarian_rmse_metres: 0.5615 - val_loss: 1.4927 - learning_rate: 0.0010
Epoch 2/100
4214/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9907 - hungarian_rmse_metres: 0.4966 - loss: 0.3888
Epoch 2: val_loss improved from 1.49275 to 1.33146, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9877 - hungarian_rmse_metres: 0.4430 - loss: 0.3592 - val_hungarian_mask_acc: 0.8274 - val_hungarian_rmse_metres: 0.5184 - val_loss: 1.3315 - learning_rate: 0.0010
Epoch 3/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9924 - hungarian_rmse_metres: 0.4408 - loss: 0.3150
Epoch 3: val_loss did not improve from 1.33146
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9894 - hungarian_rmse_metres: 0.3986 - loss: 0.2981 - val_hungarian_mask_acc: 0.7892 - val_hungarian_rmse_metres: 0.5147 - val_loss: 1.4331 - learning_rate: 0.0010
Epoch 4/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9932 - hungarian_rmse_metres: 0.4183 - loss: 0.2867
Epoch 4: val_loss improved from 1.33146 to 0.99234, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9906 - hungarian_rmse_metres: 0.3797 - loss: 0.2722 - val_hungarian_mask_acc: 0.8590 - val_hungarian_rmse_metres: 0.4982 - val_loss: 0.9923 - learning_rate: 0.0010
Epoch 5/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9940 - hungarian_rmse_metres: 0.4029 - loss: 0.2651
Epoch 5: val_loss did not improve from 0.99234
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9918 - hungarian_rmse_metres: 0.3666 - loss: 0.2521 - val_hungarian_mask_acc: 0.8476 - val_hungarian_rmse_metres: 0.5219 - val_loss: 1.1004 - learning_rate: 0.0010
Epoch 6/100
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9944 - hungarian_rmse_metres: 0.3946 - loss: 0.2538
Epoch 6: val_loss improved from 0.99234 to 0.97732, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9924 - hungarian_rmse_metres: 0.3571 - loss: 0.2381 - val_hungarian_mask_acc: 0.8627 - val_hungarian_rmse_metres: 0.4945 - val_loss: 0.9773 - learning_rate: 0.0010
Epoch 7/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9947 - hungarian_rmse_metres: 0.3840 - loss: 0.2405
Epoch 7: val_loss improved from 0.97732 to 0.88435, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9929 - hungarian_rmse_metres: 0.3477 - loss: 0.2258 - val_hungarian_mask_acc: 0.8841 - val_hungarian_rmse_metres: 0.4907 - val_loss: 0.8843 - learning_rate: 0.0010
Epoch 8/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9951 - hungarian_rmse_metres: 0.3768 - loss: 0.2315
Epoch 8: val_loss did not improve from 0.88435
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - hungarian_mask_acc: 0.9935 - hungarian_rmse_metres: 0.3421 - loss: 0.2170 - val_hungarian_mask_acc: 0.8813 - val_hungarian_rmse_metres: 0.4862 - val_loss: 0.9376 - learning_rate: 0.0010
Epoch 9/100
4218/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9952 - hungarian_rmse_metres: 0.3738 - loss: 0.2268
Epoch 9: val_loss did not improve from 0.88435
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9935 - hungarian_rmse_metres: 0.3388 - loss: 0.2132 - val_hungarian_mask_acc: 0.8962 - val_hungarian_rmse_metres: 0.4771 - val_loss: 0.9194 - learning_rate: 0.0010
Epoch 10/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9957 - hungarian_rmse_metres: 0.3672 - loss: 0.2181
Epoch 10: val_loss improved from 0.88435 to 0.75771, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9940 - hungarian_rmse_metres: 0.3330 - loss: 0.2059 - val_hungarian_mask_acc: 0.9229 - val_hungarian_rmse_metres: 0.4810 - val_loss: 0.7577 - learning_rate: 0.0010
Epoch 11/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9958 - hungarian_rmse_metres: 0.3613 - loss: 0.2108
Epoch 11: val_loss did not improve from 0.75771
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9941 - hungarian_rmse_metres: 0.3278 - loss: 0.1983 - val_hungarian_mask_acc: 0.9209 - val_hungarian_rmse_metres: 0.4782 - val_loss: 0.7818 - learning_rate: 0.0010
Epoch 12/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9960 - hungarian_rmse_metres: 0.3580 - loss: 0.2067
Epoch 12: val_loss did not improve from 0.75771
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9945 - hungarian_rmse_metres: 0.3251 - loss: 0.1942 - val_hungarian_mask_acc: 0.9198 - val_hungarian_rmse_metres: 0.4664 - val_loss: 0.7755 - learning_rate: 0.0010
Epoch 13/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9962 - hungarian_rmse_metres: 0.3567 - loss: 0.2039
Epoch 13: val_loss did not improve from 0.75771
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9950 - hungarian_rmse_metres: 0.3232 - loss: 0.1903 - val_hungarian_mask_acc: 0.9131 - val_hungarian_rmse_metres: 0.4926 - val_loss: 0.8476 - learning_rate: 0.0010
Epoch 14/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9962 - hungarian_rmse_metres: 0.3517 - loss: 0.1981
Epoch 14: val_loss did not improve from 0.75771
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9950 - hungarian_rmse_metres: 0.3190 - loss: 0.1859 - val_hungarian_mask_acc: 0.9238 - val_hungarian_rmse_metres: 0.4747 - val_loss: 0.7722 - learning_rate: 0.0010
Epoch 15/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9963 - hungarian_rmse_metres: 0.3491 - loss: 0.1962
Epoch 15: val_loss did not improve from 0.75771

Epoch 15: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - hungarian_mask_acc: 0.9949 - hungarian_rmse_metres: 0.3173 - loss: 0.1850 - val_hungarian_mask_acc: 0.9171 - val_hungarian_rmse_metres: 0.4710 - val_loss: 0.7633 - learning_rate: 0.0010
Epoch 16/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9946 - hungarian_rmse_metres: 0.3697 - loss: 0.2243
Epoch 16: val_loss improved from 0.75771 to 0.74415, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9942 - hungarian_rmse_metres: 0.3241 - loss: 0.1956 - val_hungarian_mask_acc: 0.9224 - val_hungarian_rmse_metres: 0.4590 - val_loss: 0.7442 - learning_rate: 2.0000e-04
Epoch 17/100
4210/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9954 - hungarian_rmse_metres: 0.3548 - loss: 0.2051
Epoch 17: val_loss improved from 0.74415 to 0.72394, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9948 - hungarian_rmse_metres: 0.3148 - loss: 0.1827 - val_hungarian_mask_acc: 0.9271 - val_hungarian_rmse_metres: 0.4591 - val_loss: 0.7239 - learning_rate: 2.0000e-04
Epoch 18/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9958 - hungarian_rmse_metres: 0.3496 - loss: 0.1970
Epoch 18: val_loss did not improve from 0.72394
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - hungarian_mask_acc: 0.9951 - hungarian_rmse_metres: 0.3109 - loss: 0.1770 - val_hungarian_mask_acc: 0.9243 - val_hungarian_rmse_metres: 0.4557 - val_loss: 0.7309 - learning_rate: 2.0000e-04
Epoch 19/100
4210/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9960 - hungarian_rmse_metres: 0.3453 - loss: 0.1930
Epoch 19: val_loss did not improve from 0.72394
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - hungarian_mask_acc: 0.9953 - hungarian_rmse_metres: 0.3079 - loss: 0.1740 - val_hungarian_mask_acc: 0.9146 - val_hungarian_rmse_metres: 0.4569 - val_loss: 0.7881 - learning_rate: 2.0000e-04
Epoch 20/100
4214/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9962 - hungarian_rmse_metres: 0.3430 - loss: 0.1892
Epoch 20: val_loss improved from 0.72394 to 0.68296, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - hungarian_mask_acc: 0.9955 - hungarian_rmse_metres: 0.3062 - loss: 0.1707 - val_hungarian_mask_acc: 0.9300 - val_hungarian_rmse_metres: 0.4512 - val_loss: 0.6830 - learning_rate: 2.0000e-04
Epoch 21/100
4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9963 - hungarian_rmse_metres: 0.3402 - loss: 0.1851
Epoch 21: val_loss did not improve from 0.68296
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9956 - hungarian_rmse_metres: 0.3044 - loss: 0.1677 - val_hungarian_mask_acc: 0.9096 - val_hungarian_rmse_metres: 0.4493 - val_loss: 0.7955 - learning_rate: 2.0000e-04
Epoch 22/100
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9964 - hungarian_rmse_metres: 0.3375 - loss: 0.1820
Epoch 22: val_loss did not improve from 0.68296
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9956 - hungarian_rmse_metres: 0.3022 - loss: 0.1650 - val_hungarian_mask_acc: 0.8907 - val_hungarian_rmse_metres: 0.4570 - val_loss: 0.8217 - learning_rate: 2.0000e-04
Epoch 23/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9964 - hungarian_rmse_metres: 0.3367 - loss: 0.1814
Epoch 23: val_loss did not improve from 0.68296
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9956 - hungarian_rmse_metres: 0.3019 - loss: 0.1653 - val_hungarian_mask_acc: 0.9231 - val_hungarian_rmse_metres: 0.4527 - val_loss: 0.7192 - learning_rate: 2.0000e-04
Epoch 24/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9965 - hungarian_rmse_metres: 0.3342 - loss: 0.1784
Epoch 24: val_loss did not improve from 0.68296
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9957 - hungarian_rmse_metres: 0.2994 - loss: 0.1620 - val_hungarian_mask_acc: 0.8875 - val_hungarian_rmse_metres: 0.4514 - val_loss: 0.8303 - learning_rate: 2.0000e-04
Epoch 25/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9966 - hungarian_rmse_metres: 0.3331 - loss: 0.1771
Epoch 25: val_loss did not improve from 0.68296

Epoch 25: ReduceLROnPlateau reducing learning rate to 4.0000001899898055e-05.
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9959 - hungarian_rmse_metres: 0.2986 - loss: 0.1615 - val_hungarian_mask_acc: 0.9187 - val_hungarian_rmse_metres: 0.4530 - val_loss: 0.7481 - learning_rate: 2.0000e-04
Epoch 26/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9914 - hungarian_rmse_metres: 0.3600 - loss: 0.2241
Epoch 26: val_loss improved from 0.68296 to 0.63914, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9916 - hungarian_rmse_metres: 0.3156 - loss: 0.1951 - val_hungarian_mask_acc: 0.9368 - val_hungarian_rmse_metres: 0.4490 - val_loss: 0.6391 - learning_rate: 4.0000e-05
Epoch 27/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9914 - hungarian_rmse_metres: 0.3561 - loss: 0.2188
Epoch 27: val_loss improved from 0.63914 to 0.61912, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9912 - hungarian_rmse_metres: 0.3138 - loss: 0.1923 - val_hungarian_mask_acc: 0.9384 - val_hungarian_rmse_metres: 0.4454 - val_loss: 0.6191 - learning_rate: 4.0000e-05
Epoch 28/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9920 - hungarian_rmse_metres: 0.3537 - loss: 0.2132
Epoch 28: val_loss did not improve from 0.61912
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9918 - hungarian_rmse_metres: 0.3126 - loss: 0.1887 - val_hungarian_mask_acc: 0.9384 - val_hungarian_rmse_metres: 0.4458 - val_loss: 0.6220 - learning_rate: 4.0000e-05
Epoch 29/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9923 - hungarian_rmse_metres: 0.3516 - loss: 0.2102
Epoch 29: val_loss improved from 0.61912 to 0.60636, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9920 - hungarian_rmse_metres: 0.3120 - loss: 0.1877 - val_hungarian_mask_acc: 0.9413 - val_hungarian_rmse_metres: 0.4447 - val_loss: 0.6064 - learning_rate: 4.0000e-05
Epoch 30/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9925 - hungarian_rmse_metres: 0.3505 - loss: 0.2082
Epoch 30: val_loss improved from 0.60636 to 0.60391, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9922 - hungarian_rmse_metres: 0.3108 - loss: 0.1858 - val_hungarian_mask_acc: 0.9418 - val_hungarian_rmse_metres: 0.4432 - val_loss: 0.6039 - learning_rate: 4.0000e-05
Epoch 31/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9929 - hungarian_rmse_metres: 0.3482 - loss: 0.2048
Epoch 31: val_loss improved from 0.60391 to 0.60153, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9926 - hungarian_rmse_metres: 0.3095 - loss: 0.1834 - val_hungarian_mask_acc: 0.9420 - val_hungarian_rmse_metres: 0.4439 - val_loss: 0.6015 - learning_rate: 4.0000e-05
Epoch 32/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9934 - hungarian_rmse_metres: 0.3473 - loss: 0.2022
Epoch 32: val_loss improved from 0.60153 to 0.59613, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9928 - hungarian_rmse_metres: 0.3091 - loss: 0.1824 - val_hungarian_mask_acc: 0.9430 - val_hungarian_rmse_metres: 0.4419 - val_loss: 0.5961 - learning_rate: 4.0000e-05
Epoch 33/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9933 - hungarian_rmse_metres: 0.3473 - loss: 0.2027
Epoch 33: val_loss did not improve from 0.59613
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - hungarian_mask_acc: 0.9928 - hungarian_rmse_metres: 0.3086 - loss: 0.1822 - val_hungarian_mask_acc: 0.9423 - val_hungarian_rmse_metres: 0.4435 - val_loss: 0.5979 - learning_rate: 4.0000e-05
Epoch 34/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9934 - hungarian_rmse_metres: 0.3463 - loss: 0.2008
Epoch 34: val_loss improved from 0.59613 to 0.59198, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9928 - hungarian_rmse_metres: 0.3086 - loss: 0.1814 - val_hungarian_mask_acc: 0.9432 - val_hungarian_rmse_metres: 0.4418 - val_loss: 0.5920 - learning_rate: 4.0000e-05
Epoch 35/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9937 - hungarian_rmse_metres: 0.3451 - loss: 0.1989
Epoch 35: val_loss improved from 0.59198 to 0.57559, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9931 - hungarian_rmse_metres: 0.3076 - loss: 0.1801 - val_hungarian_mask_acc: 0.9461 - val_hungarian_rmse_metres: 0.4431 - val_loss: 0.5756 - learning_rate: 4.0000e-05
Epoch 36/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9939 - hungarian_rmse_metres: 0.3447 - loss: 0.1984
Epoch 36: val_loss did not improve from 0.57559
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - hungarian_mask_acc: 0.9931 - hungarian_rmse_metres: 0.3072 - loss: 0.1796 - val_hungarian_mask_acc: 0.9427 - val_hungarian_rmse_metres: 0.4407 - val_loss: 0.5959 - learning_rate: 4.0000e-05
Epoch 37/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9938 - hungarian_rmse_metres: 0.3435 - loss: 0.1973
Epoch 37: val_loss did not improve from 0.57559
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9932 - hungarian_rmse_metres: 0.3063 - loss: 0.1785 - val_hungarian_mask_acc: 0.9450 - val_hungarian_rmse_metres: 0.4400 - val_loss: 0.5778 - learning_rate: 4.0000e-05
Epoch 38/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9938 - hungarian_rmse_metres: 0.3433 - loss: 0.1966
Epoch 38: val_loss improved from 0.57559 to 0.57293, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9931 - hungarian_rmse_metres: 0.3061 - loss: 0.1782 - val_hungarian_mask_acc: 0.9466 - val_hungarian_rmse_metres: 0.4409 - val_loss: 0.5729 - learning_rate: 4.0000e-05
Epoch 39/100
4214/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9939 - hungarian_rmse_metres: 0.3438 - loss: 0.1972
Epoch 39: val_loss did not improve from 0.57293
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9933 - hungarian_rmse_metres: 0.3063 - loss: 0.1780 - val_hungarian_mask_acc: 0.9440 - val_hungarian_rmse_metres: 0.4405 - val_loss: 0.5870 - learning_rate: 4.0000e-05
Epoch 40/100
4214/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9941 - hungarian_rmse_metres: 0.3424 - loss: 0.1952
Epoch 40: val_loss did not improve from 0.57293
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9935 - hungarian_rmse_metres: 0.3053 - loss: 0.1762 - val_hungarian_mask_acc: 0.9442 - val_hungarian_rmse_metres: 0.4380 - val_loss: 0.5835 - learning_rate: 4.0000e-05
Epoch 41/100
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9941 - hungarian_rmse_metres: 0.3413 - loss: 0.1934
Epoch 41: val_loss did not improve from 0.57293
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9934 - hungarian_rmse_metres: 0.3048 - loss: 0.1758 - val_hungarian_mask_acc: 0.9451 - val_hungarian_rmse_metres: 0.4388 - val_loss: 0.5793 - learning_rate: 4.0000e-05
Epoch 42/100
4218/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9941 - hungarian_rmse_metres: 0.3419 - loss: 0.1943
Epoch 42: val_loss improved from 0.57293 to 0.57217, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9934 - hungarian_rmse_metres: 0.3052 - loss: 0.1760 - val_hungarian_mask_acc: 0.9462 - val_hungarian_rmse_metres: 0.4380 - val_loss: 0.5722 - learning_rate: 4.0000e-05
Epoch 43/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9943 - hungarian_rmse_metres: 0.3403 - loss: 0.1919
Epoch 43: val_loss did not improve from 0.57217
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9938 - hungarian_rmse_metres: 0.3042 - loss: 0.1741 - val_hungarian_mask_acc: 0.9452 - val_hungarian_rmse_metres: 0.4380 - val_loss: 0.5798 - learning_rate: 4.0000e-05
Epoch 44/100
4218/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9944 - hungarian_rmse_metres: 0.3405 - loss: 0.1919
Epoch 44: val_loss did not improve from 0.57217
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9937 - hungarian_rmse_metres: 0.3042 - loss: 0.1741 - val_hungarian_mask_acc: 0.9451 - val_hungarian_rmse_metres: 0.4387 - val_loss: 0.5787 - learning_rate: 4.0000e-05
Epoch 45/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9946 - hungarian_rmse_metres: 0.3401 - loss: 0.1912
Epoch 45: val_loss did not improve from 0.57217
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9939 - hungarian_rmse_metres: 0.3039 - loss: 0.1737 - val_hungarian_mask_acc: 0.9442 - val_hungarian_rmse_metres: 0.4382 - val_loss: 0.5858 - learning_rate: 4.0000e-05
Epoch 46/100
4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9944 - hungarian_rmse_metres: 0.3395 - loss: 0.1907
Epoch 46: val_loss did not improve from 0.57217
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - hungarian_mask_acc: 0.9937 - hungarian_rmse_metres: 0.3036 - loss: 0.1730 - val_hungarian_mask_acc: 0.9466 - val_hungarian_rmse_metres: 0.4370 - val_loss: 0.5723 - learning_rate: 4.0000e-05
Epoch 47/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9946 - hungarian_rmse_metres: 0.3396 - loss: 0.1901
Epoch 47: val_loss did not improve from 0.57217

Epoch 47: ReduceLROnPlateau reducing learning rate to 8.000000525498762e-06.
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9938 - hungarian_rmse_metres: 0.3031 - loss: 0.1725 - val_hungarian_mask_acc: 0.9449 - val_hungarian_rmse_metres: 0.4365 - val_loss: 0.5787 - learning_rate: 4.0000e-05
Epoch 48/100
4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9844 - hungarian_rmse_metres: 0.3635 - loss: 0.2527
Epoch 48: val_loss improved from 0.57217 to 0.55784, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9824 - hungarian_rmse_metres: 0.3192 - loss: 0.2316 - val_hungarian_mask_acc: 0.9454 - val_hungarian_rmse_metres: 0.4278 - val_loss: 0.5578 - learning_rate: 8.0000e-06
Epoch 49/100
4214/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9833 - hungarian_rmse_metres: 0.3581 - loss: 0.2494
Epoch 49: val_loss improved from 0.55784 to 0.54916, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9832 - hungarian_rmse_metres: 0.3159 - loss: 0.2244 - val_hungarian_mask_acc: 0.9466 - val_hungarian_rmse_metres: 0.4267 - val_loss: 0.5492 - learning_rate: 8.0000e-06
Epoch 50/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9843 - hungarian_rmse_metres: 0.3566 - loss: 0.2444
Epoch 50: val_loss improved from 0.54916 to 0.54372, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9842 - hungarian_rmse_metres: 0.3152 - loss: 0.2196 - val_hungarian_mask_acc: 0.9473 - val_hungarian_rmse_metres: 0.4263 - val_loss: 0.5437 - learning_rate: 8.0000e-06
Epoch 51/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9848 - hungarian_rmse_metres: 0.3555 - loss: 0.2403
Epoch 51: val_loss did not improve from 0.54372
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9847 - hungarian_rmse_metres: 0.3146 - loss: 0.2167 - val_hungarian_mask_acc: 0.9470 - val_hungarian_rmse_metres: 0.4259 - val_loss: 0.5442 - learning_rate: 8.0000e-06
Epoch 52/100
4218/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9851 - hungarian_rmse_metres: 0.3543 - loss: 0.2388
Epoch 52: val_loss improved from 0.54372 to 0.54060, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9849 - hungarian_rmse_metres: 0.3142 - loss: 0.2162 - val_hungarian_mask_acc: 0.9476 - val_hungarian_rmse_metres: 0.4262 - val_loss: 0.5406 - learning_rate: 8.0000e-06
Epoch 53/100
4218/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9853 - hungarian_rmse_metres: 0.3535 - loss: 0.2368
Epoch 53: val_loss improved from 0.54060 to 0.53979, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9851 - hungarian_rmse_metres: 0.3135 - loss: 0.2144 - val_hungarian_mask_acc: 0.9474 - val_hungarian_rmse_metres: 0.4259 - val_loss: 0.5398 - learning_rate: 8.0000e-06
Epoch 54/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9859 - hungarian_rmse_metres: 0.3534 - loss: 0.2351
Epoch 54: val_loss improved from 0.53979 to 0.53669, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9855 - hungarian_rmse_metres: 0.3139 - loss: 0.2132 - val_hungarian_mask_acc: 0.9480 - val_hungarian_rmse_metres: 0.4259 - val_loss: 0.5367 - learning_rate: 8.0000e-06
Epoch 55/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9862 - hungarian_rmse_metres: 0.3538 - loss: 0.2337
Epoch 55: val_loss improved from 0.53669 to 0.53164, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9858 - hungarian_rmse_metres: 0.3140 - loss: 0.2126 - val_hungarian_mask_acc: 0.9489 - val_hungarian_rmse_metres: 0.4251 - val_loss: 0.5316 - learning_rate: 8.0000e-06
Epoch 56/100
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9862 - hungarian_rmse_metres: 0.3528 - loss: 0.2331
Epoch 56: val_loss did not improve from 0.53164
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9857 - hungarian_rmse_metres: 0.3137 - loss: 0.2121 - val_hungarian_mask_acc: 0.9481 - val_hungarian_rmse_metres: 0.4265 - val_loss: 0.5365 - learning_rate: 8.0000e-06
Epoch 57/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9865 - hungarian_rmse_metres: 0.3530 - loss: 0.2314
Epoch 57: val_loss did not improve from 0.53164
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9860 - hungarian_rmse_metres: 0.3137 - loss: 0.2112 - val_hungarian_mask_acc: 0.9484 - val_hungarian_rmse_metres: 0.4259 - val_loss: 0.5352 - learning_rate: 8.0000e-06
Epoch 58/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9867 - hungarian_rmse_metres: 0.3532 - loss: 0.2310
Epoch 58: val_loss did not improve from 0.53164
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9863 - hungarian_rmse_metres: 0.3136 - loss: 0.2098 - val_hungarian_mask_acc: 0.9484 - val_hungarian_rmse_metres: 0.4261 - val_loss: 0.5338 - learning_rate: 8.0000e-06
Epoch 59/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9866 - hungarian_rmse_metres: 0.3529 - loss: 0.2316
Epoch 59: val_loss did not improve from 0.53164
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9861 - hungarian_rmse_metres: 0.3135 - loss: 0.2111 - val_hungarian_mask_acc: 0.9492 - val_hungarian_rmse_metres: 0.4257 - val_loss: 0.5321 - learning_rate: 8.0000e-06
Epoch 60/100
4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9871 - hungarian_rmse_metres: 0.3531 - loss: 0.2305
Epoch 60: val_loss improved from 0.53164 to 0.52710, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9864 - hungarian_rmse_metres: 0.3137 - loss: 0.2104 - val_hungarian_mask_acc: 0.9500 - val_hungarian_rmse_metres: 0.4248 - val_loss: 0.5271 - learning_rate: 8.0000e-06
Epoch 61/100
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9872 - hungarian_rmse_metres: 0.3517 - loss: 0.2287
Epoch 61: val_loss did not improve from 0.52710
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9865 - hungarian_rmse_metres: 0.3129 - loss: 0.2083 - val_hungarian_mask_acc: 0.9497 - val_hungarian_rmse_metres: 0.4258 - val_loss: 0.5300 - learning_rate: 8.0000e-06
Epoch 62/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9874 - hungarian_rmse_metres: 0.3525 - loss: 0.2288
Epoch 62: val_loss improved from 0.52710 to 0.52610, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9868 - hungarian_rmse_metres: 0.3134 - loss: 0.2081 - val_hungarian_mask_acc: 0.9503 - val_hungarian_rmse_metres: 0.4256 - val_loss: 0.5261 - learning_rate: 8.0000e-06
Epoch 63/100
4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9873 - hungarian_rmse_metres: 0.3519 - loss: 0.2280
Epoch 63: val_loss improved from 0.52610 to 0.52508, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9868 - hungarian_rmse_metres: 0.3131 - loss: 0.2077 - val_hungarian_mask_acc: 0.9505 - val_hungarian_rmse_metres: 0.4255 - val_loss: 0.5251 - learning_rate: 8.0000e-06
Epoch 64/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9873 - hungarian_rmse_metres: 0.3529 - loss: 0.2286
Epoch 64: val_loss improved from 0.52508 to 0.52326, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9868 - hungarian_rmse_metres: 0.3137 - loss: 0.2081 - val_hungarian_mask_acc: 0.9507 - val_hungarian_rmse_metres: 0.4252 - val_loss: 0.5233 - learning_rate: 8.0000e-06
Epoch 65/100
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9875 - hungarian_rmse_metres: 0.3517 - loss: 0.2272
Epoch 65: val_loss improved from 0.52326 to 0.51796, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9869 - hungarian_rmse_metres: 0.3133 - loss: 0.2079 - val_hungarian_mask_acc: 0.9520 - val_hungarian_rmse_metres: 0.4251 - val_loss: 0.5180 - learning_rate: 8.0000e-06
Epoch 66/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9879 - hungarian_rmse_metres: 0.3515 - loss: 0.2256
Epoch 66: val_loss did not improve from 0.51796
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9872 - hungarian_rmse_metres: 0.3131 - loss: 0.2062 - val_hungarian_mask_acc: 0.9516 - val_hungarian_rmse_metres: 0.4253 - val_loss: 0.5198 - learning_rate: 8.0000e-06
Epoch 67/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9880 - hungarian_rmse_metres: 0.3520 - loss: 0.2252
Epoch 67: val_loss did not improve from 0.51796
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9871 - hungarian_rmse_metres: 0.3134 - loss: 0.2058 - val_hungarian_mask_acc: 0.9518 - val_hungarian_rmse_metres: 0.4250 - val_loss: 0.5191 - learning_rate: 8.0000e-06
Epoch 68/100
4218/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9882 - hungarian_rmse_metres: 0.3514 - loss: 0.2245
Epoch 68: val_loss improved from 0.51796 to 0.51635, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - hungarian_mask_acc: 0.9874 - hungarian_rmse_metres: 0.3129 - loss: 0.2050 - val_hungarian_mask_acc: 0.9525 - val_hungarian_rmse_metres: 0.4256 - val_loss: 0.5163 - learning_rate: 8.0000e-06
Epoch 69/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9881 - hungarian_rmse_metres: 0.3518 - loss: 0.2249
Epoch 69: val_loss did not improve from 0.51635
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9872 - hungarian_rmse_metres: 0.3134 - loss: 0.2059 - val_hungarian_mask_acc: 0.9523 - val_hungarian_rmse_metres: 0.4257 - val_loss: 0.5182 - learning_rate: 8.0000e-06
Epoch 70/100
4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9882 - hungarian_rmse_metres: 0.3516 - loss: 0.2244
Epoch 70: val_loss did not improve from 0.51635
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9874 - hungarian_rmse_metres: 0.3134 - loss: 0.2056 - val_hungarian_mask_acc: 0.9518 - val_hungarian_rmse_metres: 0.4262 - val_loss: 0.5205 - learning_rate: 8.0000e-06
Epoch 71/100
4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9883 - hungarian_rmse_metres: 0.3513 - loss: 0.2239
Epoch 71: val_loss did not improve from 0.51635
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9876 - hungarian_rmse_metres: 0.3130 - loss: 0.2043 - val_hungarian_mask_acc: 0.9521 - val_hungarian_rmse_metres: 0.4260 - val_loss: 0.5196 - learning_rate: 8.0000e-06
Epoch 72/100
4214/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9884 - hungarian_rmse_metres: 0.3517 - loss: 0.2236
Epoch 72: val_loss improved from 0.51635 to 0.51323, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9877 - hungarian_rmse_metres: 0.3129 - loss: 0.2038 - val_hungarian_mask_acc: 0.9530 - val_hungarian_rmse_metres: 0.4247 - val_loss: 0.5132 - learning_rate: 8.0000e-06
Epoch 73/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9884 - hungarian_rmse_metres: 0.3514 - loss: 0.2230
Epoch 73: val_loss did not improve from 0.51323
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9876 - hungarian_rmse_metres: 0.3130 - loss: 0.2043 - val_hungarian_mask_acc: 0.9523 - val_hungarian_rmse_metres: 0.4255 - val_loss: 0.5177 - learning_rate: 8.0000e-06
Epoch 74/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9887 - hungarian_rmse_metres: 0.3506 - loss: 0.2212
Epoch 74: val_loss did not improve from 0.51323
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9879 - hungarian_rmse_metres: 0.3126 - loss: 0.2027 - val_hungarian_mask_acc: 0.9522 - val_hungarian_rmse_metres: 0.4256 - val_loss: 0.5191 - learning_rate: 8.0000e-06
Epoch 75/100
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9886 - hungarian_rmse_metres: 0.3511 - loss: 0.2230
Epoch 75: val_loss did not improve from 0.51323
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9878 - hungarian_rmse_metres: 0.3133 - loss: 0.2042 - val_hungarian_mask_acc: 0.9523 - val_hungarian_rmse_metres: 0.4249 - val_loss: 0.5160 - learning_rate: 8.0000e-06
Epoch 76/100
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9891 - hungarian_rmse_metres: 0.3512 - loss: 0.2215
Epoch 76: val_loss did not improve from 0.51323
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9882 - hungarian_rmse_metres: 0.3130 - loss: 0.2023 - val_hungarian_mask_acc: 0.9527 - val_hungarian_rmse_metres: 0.4256 - val_loss: 0.5156 - learning_rate: 8.0000e-06
Epoch 77/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9890 - hungarian_rmse_metres: 0.3515 - loss: 0.2214
Epoch 77: val_loss did not improve from 0.51323

Epoch 77: ReduceLROnPlateau reducing learning rate to 1.6000001778593287e-06.
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9881 - hungarian_rmse_metres: 0.3134 - loss: 0.2030 - val_hungarian_mask_acc: 0.9526 - val_hungarian_rmse_metres: 0.4252 - val_loss: 0.5167 - learning_rate: 8.0000e-06
Epoch 78/100
4214/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9804 - hungarian_rmse_metres: 0.3628 - loss: 0.2643
Epoch 78: val_loss improved from 0.51323 to 0.48829, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9806 - hungarian_rmse_metres: 0.3199 - loss: 0.2380 - val_hungarian_mask_acc: 0.9553 - val_hungarian_rmse_metres: 0.4178 - val_loss: 0.4883 - learning_rate: 1.6000e-06
Epoch 79/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9837 - hungarian_rmse_metres: 0.3581 - loss: 0.2477
Epoch 79: val_loss improved from 0.48829 to 0.48785, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9824 - hungarian_rmse_metres: 0.3179 - loss: 0.2292 - val_hungarian_mask_acc: 0.9537 - val_hungarian_rmse_metres: 0.4165 - val_loss: 0.4879 - learning_rate: 1.6000e-06
Epoch 80/100
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9841 - hungarian_rmse_metres: 0.3574 - loss: 0.2457
Epoch 80: val_loss improved from 0.48785 to 0.48524, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - hungarian_mask_acc: 0.9829 - hungarian_rmse_metres: 0.3176 - loss: 0.2267 - val_hungarian_mask_acc: 0.9531 - val_hungarian_rmse_metres: 0.4157 - val_loss: 0.4852 - learning_rate: 1.6000e-06
Epoch 81/100
4218/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9839 - hungarian_rmse_metres: 0.3566 - loss: 0.2445
Epoch 81: val_loss improved from 0.48524 to 0.48470, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - hungarian_mask_acc: 0.9829 - hungarian_rmse_metres: 0.3173 - loss: 0.2258 - val_hungarian_mask_acc: 0.9521 - val_hungarian_rmse_metres: 0.4152 - val_loss: 0.4847 - learning_rate: 1.6000e-06
Epoch 82/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9842 - hungarian_rmse_metres: 0.3565 - loss: 0.2439
Epoch 82: val_loss improved from 0.48470 to 0.48284, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - hungarian_mask_acc: 0.9830 - hungarian_rmse_metres: 0.3175 - loss: 0.2257 - val_hungarian_mask_acc: 0.9526 - val_hungarian_rmse_metres: 0.4151 - val_loss: 0.4828 - learning_rate: 1.6000e-06
Epoch 83/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9843 - hungarian_rmse_metres: 0.3564 - loss: 0.2432
Epoch 83: val_loss did not improve from 0.48284
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9834 - hungarian_rmse_metres: 0.3172 - loss: 0.2246 - val_hungarian_mask_acc: 0.9523 - val_hungarian_rmse_metres: 0.4148 - val_loss: 0.4829 - learning_rate: 1.6000e-06
Epoch 84/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9844 - hungarian_rmse_metres: 0.3564 - loss: 0.2426
Epoch 84: val_loss did not improve from 0.48284
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9833 - hungarian_rmse_metres: 0.3173 - loss: 0.2241 - val_hungarian_mask_acc: 0.9519 - val_hungarian_rmse_metres: 0.4148 - val_loss: 0.4828 - learning_rate: 1.6000e-06
Epoch 85/100
4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9844 - hungarian_rmse_metres: 0.3562 - loss: 0.2430
Epoch 85: val_loss did not improve from 0.48284
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9835 - hungarian_rmse_metres: 0.3172 - loss: 0.2242 - val_hungarian_mask_acc: 0.9513 - val_hungarian_rmse_metres: 0.4151 - val_loss: 0.4836 - learning_rate: 1.6000e-06
Epoch 86/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9845 - hungarian_rmse_metres: 0.3558 - loss: 0.2418
Epoch 86: val_loss did not improve from 0.48284
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9836 - hungarian_rmse_metres: 0.3169 - loss: 0.2231 - val_hungarian_mask_acc: 0.9502 - val_hungarian_rmse_metres: 0.4149 - val_loss: 0.4847 - learning_rate: 1.6000e-06
Epoch 87/100
4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9847 - hungarian_rmse_metres: 0.3560 - loss: 0.2416
Epoch 87: val_loss did not improve from 0.48284

Epoch 87: ReduceLROnPlateau reducing learning rate to 1e-06.
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9837 - hungarian_rmse_metres: 0.3168 - loss: 0.2227 - val_hungarian_mask_acc: 0.9510 - val_hungarian_rmse_metres: 0.4149 - val_loss: 0.4841 - learning_rate: 1.6000e-06
Epoch 88/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9833 - hungarian_rmse_metres: 0.3571 - loss: 0.2464
Epoch 88: val_loss improved from 0.48284 to 0.47991, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9832 - hungarian_rmse_metres: 0.3178 - loss: 0.2248 - val_hungarian_mask_acc: 0.9500 - val_hungarian_rmse_metres: 0.4141 - val_loss: 0.4799 - learning_rate: 1.0000e-06
Epoch 89/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9840 - hungarian_rmse_metres: 0.3562 - loss: 0.2440
Epoch 89: val_loss improved from 0.47991 to 0.47840, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9832 - hungarian_rmse_metres: 0.3176 - loss: 0.2250 - val_hungarian_mask_acc: 0.9497 - val_hungarian_rmse_metres: 0.4137 - val_loss: 0.4784 - learning_rate: 1.0000e-06
Epoch 90/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9844 - hungarian_rmse_metres: 0.3562 - loss: 0.2421
Epoch 90: val_loss improved from 0.47840 to 0.47659, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9834 - hungarian_rmse_metres: 0.3175 - loss: 0.2245 - val_hungarian_mask_acc: 0.9506 - val_hungarian_rmse_metres: 0.4137 - val_loss: 0.4766 - learning_rate: 1.0000e-06
Epoch 91/100
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9844 - hungarian_rmse_metres: 0.3561 - loss: 0.2419
Epoch 91: val_loss improved from 0.47659 to 0.47650, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9834 - hungarian_rmse_metres: 0.3173 - loss: 0.2244 - val_hungarian_mask_acc: 0.9502 - val_hungarian_rmse_metres: 0.4137 - val_loss: 0.4765 - learning_rate: 1.0000e-06
Epoch 92/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9847 - hungarian_rmse_metres: 0.3560 - loss: 0.2411
Epoch 92: val_loss did not improve from 0.47650
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - hungarian_mask_acc: 0.9836 - hungarian_rmse_metres: 0.3173 - loss: 0.2232 - val_hungarian_mask_acc: 0.9503 - val_hungarian_rmse_metres: 0.4135 - val_loss: 0.4769 - learning_rate: 1.0000e-06
Epoch 93/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9845 - hungarian_rmse_metres: 0.3554 - loss: 0.2417
Epoch 93: val_loss did not improve from 0.47650
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9837 - hungarian_rmse_metres: 0.3173 - loss: 0.2233 - val_hungarian_mask_acc: 0.9502 - val_hungarian_rmse_metres: 0.4138 - val_loss: 0.4772 - learning_rate: 1.0000e-06
Epoch 94/100
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9847 - hungarian_rmse_metres: 0.3558 - loss: 0.2416
Epoch 94: val_loss improved from 0.47650 to 0.47570, saving model to norm+depth+noise_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9837 - hungarian_rmse_metres: 0.3176 - loss: 0.2239 - val_hungarian_mask_acc: 0.9504 - val_hungarian_rmse_metres: 0.4136 - val_loss: 0.4757 - learning_rate: 1.0000e-06
Epoch 95/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9848 - hungarian_rmse_metres: 0.3558 - loss: 0.2412
Epoch 95: val_loss did not improve from 0.47570
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - hungarian_mask_acc: 0.9839 - hungarian_rmse_metres: 0.3176 - loss: 0.2230 - val_hungarian_mask_acc: 0.9505 - val_hungarian_rmse_metres: 0.4138 - val_loss: 0.4766 - learning_rate: 1.0000e-06
Epoch 96/100
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9847 - hungarian_rmse_metres: 0.3569 - loss: 0.2428
Epoch 96: val_loss did not improve from 0.47570
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - hungarian_mask_acc: 0.9837 - hungarian_rmse_metres: 0.3181 - loss: 0.2241 - val_hungarian_mask_acc: 0.9503 - val_hungarian_rmse_metres: 0.4136 - val_loss: 0.4769 - learning_rate: 1.0000e-06
Epoch 97/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9844 - hungarian_rmse_metres: 0.3558 - loss: 0.2417
Epoch 97: val_loss did not improve from 0.47570
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9836 - hungarian_rmse_metres: 0.3173 - loss: 0.2233 - val_hungarian_mask_acc: 0.9499 - val_hungarian_rmse_metres: 0.4137 - val_loss: 0.4780 - learning_rate: 1.0000e-06
Epoch 98/100
4216/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9850 - hungarian_rmse_metres: 0.3559 - loss: 0.2409
Epoch 98: val_loss did not improve from 0.47570
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9841 - hungarian_rmse_metres: 0.3176 - loss: 0.2224 - val_hungarian_mask_acc: 0.9504 - val_hungarian_rmse_metres: 0.4139 - val_loss: 0.4784 - learning_rate: 1.0000e-06
Epoch 99/100
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9847 - hungarian_rmse_metres: 0.3562 - loss: 0.2416
Epoch 99: val_loss did not improve from 0.47570
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - hungarian_mask_acc: 0.9838 - hungarian_rmse_metres: 0.3176 - loss: 0.2230 - val_hungarian_mask_acc: 0.9502 - val_hungarian_rmse_metres: 0.4138 - val_loss: 0.4773 - learning_rate: 1.0000e-06
Epoch 100/100
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9846 - hungarian_rmse_metres: 0.3557 - loss: 0.2407
Epoch 100: val_loss did not improve from 0.47570
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9836 - hungarian_rmse_metres: 0.3173 - loss: 0.2227 - val_hungarian_mask_acc: 0.9493 - val_hungarian_rmse_metres: 0.4136 - val_loss: 0.4774 - learning_rate: 1.0000e-06
Restoring model weights from the end of the best epoch: 94.
--- ADDESTRAMENTO COMPLETATO ---

1- AGGIUNGI GAUSSIAN NOISE
2- RADAR DROPOUT
3- SLIDING WINDOW? 
4- DIGITAL BEAMFOMING? 